In [30]:
!pip install datasets pandas
from sklearn.model_selection import train_test_split
import pandas as pd

In [31]:
from datasets import load_dataset

In [32]:
!pip install -q pandas pyarrow

import pandas as pd

# Download FEVER training dataset in Parquet format
train_url = "https://huggingface.co/datasets/fever/fever/resolve/1c55aad95efe327fc6e71a5893d6040d93a8f98c/v1.0/fever-train.parquet"

df = pd.read_parquet(train_url)

print("FEVER dataset downloaded successfully!")
print("Total records:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

FEVER dataset downloaded successfully!
Total records: 311431

Columns:
['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id']

First 5 records:


,id,label,claim,evidence_annotation_id,evidence_id,evidence_wiki_url,evidence_sentence_id
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,92206,104971,Nikolaj_Coster-Waldau,7
1,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,92206,104971,Fox_Broadcasting_Company,-1
2,150448,SUPPORTS,Roman Atwood is a content creator.,174271,187498,Roman_Atwood,1
3,150448,SUPPORTS,Roman Atwood is a content creator.,174271,187499,Roman_Atwood,3
4,214861,SUPPORTS,"History of art includes architecture, dance, s...",255136,254645,History_of_art,2


In [33]:
# 1. Dataset size
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

Total rows: 311431
Total columns: 7


In [34]:
# 2. Label distribution
print("\nLabel Distribution:")
print(df["label"].value_counts())


Label Distribution:
label
SUPPORTS           193756
REFUTES             70066
NOT ENOUGH INFO     47609
Name: count, dtype: int64


In [35]:
# 3. Missing values
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0
dtype: int64


In [36]:
# 4. Unique claims
print("\nUnique Claim IDs:", df["id"].nunique())



Unique Claim IDs: 145449


In [37]:

# 5. Duplicate claim text
print("\nUnique Claim Text:", df["claim"].nunique())



Unique Claim Text: 135545


In [38]:
# 6. Duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())


Duplicate Rows: 6


In [39]:
# 7. Multiple evidence references
print("\nRows per Claim ID:")
print(df.groupby("id").size().describe())


Rows per Claim ID:
count    145449.000000
mean          2.141170
std           3.663248
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         338.000000
dtype: float64


In [40]:
# Label distribution
print("Label Distribution:")
print(df["label"].value_counts())

print("\nLabel Distribution Percentage:")
print((df["label"].value_counts(normalize=True) * 100).round(2))

Label Distribution:
label
SUPPORTS           193756
REFUTES             70066
NOT ENOUGH INFO     47609
Name: count, dtype: int64

Label Distribution Percentage:
label
SUPPORTS           62.21
REFUTES            22.50
NOT ENOUGH INFO    15.29
Name: proportion, dtype: float64


In [41]:
# Missing values
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0
dtype: int64


In [42]:
# Exact duplicates
print("\nExact duplicate rows:", df.duplicated().sum())


Exact duplicate rows: 6


In [43]:
# Missing claim text
print("\nMissing claim text:", df["claim"].isnull().sum())



Missing claim text: 0


In [44]:
# Empty claims
print("\nEmpty claims:", (df["claim"].astype(str).str.strip() == "").sum())


Empty claims: 0


In [45]:
print("Label Counts:")
print(df["label"].value_counts())

print("\nLabel Percentage:")
print((df["label"].value_counts(normalize=True) * 100).round(2))

Label Counts:
label
SUPPORTS           193756
REFUTES             70066
NOT ENOUGH INFO     47609
Name: count, dtype: int64

Label Percentage:
label
SUPPORTS           62.21
REFUTES            22.50
NOT ENOUGH INFO    15.29
Name: proportion, dtype: float64


In [46]:
# Step 1: Remove exact duplicate rows
df_clean = df.drop_duplicates().copy()


In [47]:
# Step 2: Map FEVER labels to our project labels
label_mapping = {
    "SUPPORTS": "Supported",
    "REFUTES": "Refuted",
    "NOT ENOUGH INFO": "Insufficient Information"
}

df_clean["project_label"] = df_clean["label"].map(label_mapping)

In [48]:
# Step 3: Group rows by claim ID
# This preserves multiple evidence references for the same claim

claim_level_df = df_clean.groupby("id", as_index=False).agg({
    "claim": "first",
    "label": "first",
    "project_label": "first",
    "evidence_annotation_id": "first",
    "evidence_id": lambda x: list(pd.unique(x.dropna())),
    "evidence_wiki_url": lambda x: list(pd.unique(x.dropna())),
    "evidence_sentence_id": lambda x: list(pd.unique(x.dropna()))
})

In [49]:
# Step 4: Rename ID column
claim_level_df = claim_level_df.rename(columns={"id": "claim_id"})


In [50]:
# Step 5: Display results
print("Claim-level dataset created!")
print("Total unique claims:", len(claim_level_df))

print("\nLabel distribution:")
print(claim_level_df["project_label"].value_counts())

display(claim_level_df.head())


Claim-level dataset created!
Total unique claims: 145449

Label distribution:
project_label
Supported                   80035
Insufficient Information    35639
Refuted                     29775
Name: count, dtype: int64


,claim_id,claim,label,project_label,evidence_annotation_id,evidence_id,evidence_wiki_url,evidence_sentence_id
0,3,Chris Hemsworth appeared in A Perfect Getaway.,SUPPORTS,Supported,15732,[19585],[Chris_Hemsworth],[2]
1,4,Chris Hemsworth disappeared in A Perfect Getaway.,NOT ENOUGH INFO,Insufficient Information,15230,[-1],[],[-1]
2,7,Roald Dahl is a writer.,SUPPORTS,Supported,15234,[18987],[Roald_Dahl],[-1]
3,8,Roald Dahl is a governor.,REFUTES,Refuted,15233,[18986],[Roald_Dahl],[-1]
4,9,Ireland has relatively low-lying mountains.,SUPPORTS,Supported,15235,[18988],[Ireland],[10]


In [51]:
# Create a balanced dataset
supported = claim_level_df[
    claim_level_df["project_label"] == "Supported"
].sample(n=3333, random_state=42)

refuted = claim_level_df[
    claim_level_df["project_label"] == "Refuted"
].sample(n=3333, random_state=42)

insufficient = claim_level_df[
    claim_level_df["project_label"] == "Insufficient Information"
].sample(n=3334, random_state=42)


In [52]:
# Combine all three classes
balanced_df = pd.concat(
    [supported, refuted, insufficient]
)

In [53]:
# Shuffle the dataset
balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Balanced dataset created!")
print(balanced_df["project_label"].value_counts())

Balanced dataset created!
project_label
Insufficient Information    3334
Refuted                     3333
Supported                   3333
Name: count, dtype: int64


In [54]:
# First: separate 80% training and 20% temporary data
train_df, temp_df = train_test_split(
    balanced_df,
    test_size=0.20,
    random_state=42,
    stratify=balanced_df["project_label"]
)

In [55]:
# Second: split temporary data into validation and testing
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["project_label"]
)




In [56]:

# Check the dataset sizes
print("Training dataset:", len(train_df))
print("Validation dataset:", len(val_df))
print("Testing dataset:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Training dataset: 8000
Validation dataset: 1000
Testing dataset: 1000
Total: 10000


In [57]:
# Save balanced dataset
balanced_df.to_csv("fever_balanced_10000.csv", index=False)

In [58]:

# Save training dataset
train_df.to_csv("fever_train.csv", index=False)



In [59]:
# Save validation dataset
val_df.to_csv("fever_validation.csv", index=False)

In [60]:
# Save testing dataset
test_df.to_csv("fever_test.csv", index=False)

In [61]:
print("All datasets saved successfully!")

All datasets saved successfully!


In [62]:
from google.colab import files

files.download("fever_balanced_10000.csv")
files.download("fever_train.csv")
files.download("fever_validation.csv")
files.download("fever_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>